In [1]:
FORCE_RELOAD = False
DATASET_NAME = 'DiscoveryWorld'
DATASET_VERSION = 'v2'
DELETE_IF_EXISTS_DATASET = True


In [3]:
from dataset_manager import datasetManager

df= datasetManager.load_dataset(FORCE_RELOAD)
df = df[
    (df["scenarioName"] == "Reactor Lab")
    & (df["difficultyStr"] == "Easy")
]
df
# print(df.iloc[10]['taskDescription'])

# print(df.iloc[10]['taskDescription'])

Loading dataset: dataset.parquet


,id,scenarioName,difficultyStr,seed,numUserAgents,maxAgents,taskName,taskDescription,maxScore,scorecard
10,10,Reactor Lab,Easy,0,1,3,ReactorTaskEasy,You are at the Quantum Reactor Lab on Planet X...,11,"[{'name': 'Quantum Crystals Taken', 'descripti..."


In [ ]:
from agent_runner import AgentRunner
from agents.AgentsFactory import AgentType
from llm import get_llm


result = AgentRunner().runAgent( 
                           agent = AgentType.LLMAgent, 
                           scenarioName= "Reactor Lab",
                           difficultyStr= "Easy",
                           model=get_llm("gpt-6-luna"),
                           numSteps=50,
                           )    

In [13]:
import json
from deepdiff import DeepDiff

with open("observation-0.json", "r", encoding="utf-8") as file:
    previous = json.load(file)["ui"] 
    
with open("observation-1.json", "r", encoding="utf-8") as file:
    current = json.load(file)["ui"] 
        
delta = DeepDiff(previous, current, ignore_order=True)

print(delta.pretty())

Item root['accessibleEnvironmentObjects'][0] added to iterable.
Value of root['accessibleEnvironmentObjects'][0]['description'] changed from "wall" to "rusted key (heavily rusted)".
Value of root['accessibleEnvironmentObjects'][0]['name'] changed from "wall" to "rusted key (heavily rusted)".
Value of root['accessibleEnvironmentObjects'][0]['uuid'] changed from 44861 to 33120.
Value of root['agentLocation']['faceDirection'] changed from "south" to "west".
Value of root['lastActionMessage'] changed from "" to "I rotated to face west.".
Value of root['world_steps'] changed from 1 to 2.


In [10]:
previous["ui"]["nearbyObjects"]["objects"]

{'east': [{'description': 'floor',
   'distance': 1,
   'name': 'floor',
   'uuid': 25494},
  {'description': 'floor', 'distance': 2, 'name': 'floor', 'uuid': 65141},
  {'description': 'floor', 'distance': 3, 'name': 'floor', 'uuid': 32662}],
 'north': [{'description': 'grass',
   'distance': 3,
   'name': 'grass',
   'uuid': 12855},
  {'description': 'wall', 'distance': 2, 'name': 'wall', 'uuid': 37350},
  {'description': 'table', 'distance': 1, 'name': 'table', 'uuid': 58743},
  {'description': 'Dispenser (Substance A) (on table [uuid: 58743])',
   'distance': 1,
   'name': 'Dispenser (Substance A)',
   'uuid': 21559},
  {'description': 'floor', 'distance': 1, 'name': 'floor', 'uuid': 2056}],
 'north-east': [{'description': 'grass',
   'distance': 4,
   'name': 'grass',
   'uuid': 51617},
  {'description': 'wall', 'distance': 3, 'name': 'wall', 'uuid': 1787},
  {'description': 'table', 'distance': 2, 'name': 'table', 'uuid': 31029},
  {'description': 'Dispenser (Substance B) (on tabl

In [17]:
import jsonpatch

patch = jsonpatch.make_patch(previous, current)

print(patch.to_string())

reconstructed = patch.apply(previous)

assert reconstructed == current

[{"op": "replace", "path": "/accessibleEnvironmentObjects/0/description", "value": "floor"}, {"op": "replace", "path": "/accessibleEnvironmentObjects/0/name", "value": "floor"}, {"op": "replace", "path": "/accessibleEnvironmentObjects/0/uuid", "value": 29924}, {"op": "add", "path": "/accessibleEnvironmentObjects/1", "value": {"description": "rusted key (heavily rusted)", "name": "rusted key (heavily rusted)", "uuid": 33120}}, {"op": "replace", "path": "/agentLocation/faceDirection", "value": "west"}, {"op": "replace", "path": "/lastActionMessage", "value": "I rotated to face west."}, {"op": "replace", "path": "/world_steps", "value": 2}]


In [20]:
from llm import get_llm

model = get_llm(model_name= "qwen/qwen3.8-27b")
model = get_llm(model_name= "openai/gpt-oss-120b")
    
response = model.invoke(f"Give me a reduced version of the following, only return the reduced version: {current}")
response


AIMessage(content='{\n  "accessibleEnvironmentObjects": [\n    {"name": "floor", "uuid": 29924},\n    {"name": "rusted key (heavily rusted)", "uuid": 33120},\n    {"name": "floor", "uuid": 47477}\n  ],\n  "agentLocation": {\n    "directions_blocked": ["north", "south"],\n    "directions_you_can_move": ["east", "west"],\n    "faceDirection": "west",\n    "x": 18,\n    "y": 12\n  },\n  "inventoryObjects": [],\n  "nearbyObjects": {\n    "west": [\n      {"name": "rusted key (heavily rusted)", "uuid": 33120, "distance": 1},\n      {"name": "floor", "uuid": 29924, "distance": 1}\n    ],\n    "north": [\n      {"name": "table", "uuid": 58743, "distance": 1},\n      {"name": "Dispenser (Substance A)", "uuid": 21559, "distance": 1},\n      {"name": "floor", "uuid": 2056, "distance": 1}\n    ],\n    "north-east": [\n      {"name": "table", "uuid": 31029, "distance": 2},\n      {"name": "Dispenser (Substance B)", "uuid": 57736, "distance": 2}\n    ],\n    "north-west": [\n      {"name": "table",